# Parte 3 — Ecuaciones elípticas
## 3.3 El principio del máximo
### 3.3.01 Principios débil y fuerte, unicidad y estabilidad

**Fuente principal:** parte derecha de la página 6 y comienzo de la página 7 de las notas manuscritas del archivo `Ecuación De Onda(1).pdf`.

Este notebook continúa después de `03.2.01_Propiedad_del_promedio_y_subarmonicidad_CORREGIDO.ipynb`.
Se detiene antes del teorema de comparación y de la teoría sistemática de subsoluciones y supersoluciones, que corresponden al siguiente notebook.

## Convenciones editoriales

- **Transcripción literal de las notas:** conserva el orden y la notación visible.
- **Aclaración:** explicita hipótesis necesarias que no aparecen completas en la página.
- **Corrección editorial:** corrige una formulación para que el enunciado sea matemáticamente preciso.
- **Complemento:** añade una demostración o consecuencia necesaria para el examen.
- Toda la matemática usa exclusivamente `$...$` y `$$...$$` para que se renderice correctamente en Jupyter Notebook, JupyterLab y VS Code.

# Simulaciones y gráficas automáticas

Las celdas siguientes se ejecutan directamente. No existe ninguna bandera `VIDEO=True` ni es necesario cambiar parámetros para producir las salidas.

El notebook detecta automáticamente CuPy/CUDA:

- con GPU usa una malla más fina y genera un MP4 de alta resolución a 120 fps;
- sin GPU usa una malla moderada y genera un MP4 a 60 fps;
- si FFmpeg no está disponible, guarda y muestra un GIF automáticamente.

Las simulaciones ilustran, pero **no demuestran**, los teoremas del máximo.

In [ ]:
from __future__ import annotations

import math
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

# ------------------------------------------------------------
# Rutas de salida robustas
# ------------------------------------------------------------
CWD = Path.cwd()
BASE = CWD.parent if CWD.name == "notebooks" else CWD
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Backend automático
# ------------------------------------------------------------
GPU_AVAILABLE = False
try:
    import cupy as cp
    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend numérico: CuPy/CUDA")
    else:
        xp = np
        print("Backend numérico: NumPy/CPU")
except Exception as exc:
    xp = np
    print("Backend numérico: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)


def to_cpu(array):
    """Convierte un arreglo CuPy a NumPy; deja intacto un arreglo NumPy."""
    return cp.asnumpy(array) if GPU_AVAILABLE else np.asarray(array)


def save_and_display_animation(animation, stem, fps, dpi, bitrate):
    """Guarda y muestra una animación sin requerir activar banderas."""
    if shutil.which("ffmpeg"):
        path = ANIM_DIR / f"{stem}.mp4"
        writer = FFMpegWriter(
            fps=fps,
            bitrate=bitrate,
            metadata={"title": stem},
        )
        animation.save(path, writer=writer, dpi=dpi)
        display(Video(str(path), embed=True))
    else:
        path = ANIM_DIR / f"{stem}.gif"
        animation.save(
            path,
            writer=PillowWriter(fps=min(fps, 35)),
            dpi=min(dpi, 110),
        )
        display(Image(filename=str(path)))
    print("Salida guardada en:", path.resolve())
    return path

## Simulación 3.3.A — Relajación armónica y principio débil del máximo

En el cuadrado

$$
\Omega=(0,1)^2
$$

se resuelve numéricamente

$$
\begin{cases}
\Delta u=0, & \text{en }\Omega,\\
u=g, & \text{sobre }\partial\Omega,
\end{cases}
$$

con datos de frontera no constantes. Durante la relajación se registra

$$
\max_{\Omega_h}U,
\qquad
\min_{\Omega_h}U,
$$

y se comparan con los extremos de los datos de frontera.

En el estado discreto armónico cada nodo interior es el promedio de sus cuatro vecinos. Por ello no puede superar a todos sus vecinos ni quedar por debajo de todos ellos.

In [ ]:
# ------------------------------------------------------------
# Configuración automática de calidad
# ------------------------------------------------------------
if GPU_AVAILABLE:
    N = 512
    N_ITER = 7000
    N_FRAMES = 240
    FPS = 120
    FIGSIZE = (16.0, 9.0)
    DPI = 160
    BITRATE = 18000
else:
    N = 220
    N_ITER = 2800
    N_FRAMES = 140
    FPS = 60
    FIGSIZE = (12.0, 7.0)
    DPI = 120
    BITRATE = 8000

x = xp.linspace(0.0, 1.0, N)
y = xp.linspace(0.0, 1.0, N)
h = float(to_cpu(x[1] - x[0]))

# Dos problemas de Dirichlet. El segundo es una perturbación del primero.
g_bottom_1 = 0.20 + 0.15 * xp.sin(2.0 * math.pi * x)
g_top_1 = 0.75 + 0.20 * xp.sin(math.pi * x) ** 2
g_left_1 = xp.linspace(float(to_cpu(g_bottom_1[0])), float(to_cpu(g_top_1[0])), N)
g_right_1 = xp.linspace(float(to_cpu(g_bottom_1[-1])), float(to_cpu(g_top_1[-1])), N)

perturbation = 0.08 * xp.sin(3.0 * math.pi * x) ** 2
g_bottom_2 = g_bottom_1

g_top_2 = g_top_1 + perturbation
g_left_2 = g_left_1
g_right_2 = g_right_1


def initialize_solution(g_bottom, g_top, g_left, g_right):
    U = xp.zeros((N, N), dtype=xp.float64)
    U[0, :] = g_bottom
    U[-1, :] = g_top
    U[:, 0] = g_left
    U[:, -1] = g_right
    return U


U1 = initialize_solution(g_bottom_1, g_top_1, g_left_1, g_right_1)
U2 = initialize_solution(g_bottom_2, g_top_2, g_left_2, g_right_2)

boundary_1 = xp.concatenate((g_bottom_1, g_top_1, g_left_1, g_right_1))
boundary_2 = xp.concatenate((g_bottom_2, g_top_2, g_left_2, g_right_2))
boundary_difference = xp.concatenate((
    g_bottom_1 - g_bottom_2,
    g_top_1 - g_top_2,
    g_left_1 - g_left_2,
    g_right_1 - g_right_2,
))

boundary_min_1 = float(to_cpu(xp.min(boundary_1)))
boundary_max_1 = float(to_cpu(xp.max(boundary_1)))
boundary_diff_norm = float(to_cpu(xp.max(xp.abs(boundary_difference))))

save_steps = set(
    np.unique(np.round(np.geomspace(1, N_ITER, N_FRAMES)).astype(int)).tolist()
)

snapshots = [to_cpu(U1).astype(np.float32)]
steps = [0]
interior_maxima = [float(to_cpu(xp.max(U1[1:-1, 1:-1])))]
interior_minima = [float(to_cpu(xp.min(U1[1:-1, 1:-1])))]
stability_norms = [float(to_cpu(xp.max(xp.abs(U1 - U2))))]
residuals = []


def residual_inf(U):
    R = (
        U[2:, 1:-1] + U[:-2, 1:-1]
        + U[1:-1, 2:] + U[1:-1, :-2]
        - 4.0 * U[1:-1, 1:-1]
    ) / h**2
    return float(to_cpu(xp.max(xp.abs(R))))


residuals.append(residual_inf(U1))
omega = 0.88

for k in range(1, N_ITER + 1):
    avg1 = 0.25 * (
        U1[2:, 1:-1] + U1[:-2, 1:-1]
        + U1[1:-1, 2:] + U1[1:-1, :-2]
    )
    avg2 = 0.25 * (
        U2[2:, 1:-1] + U2[:-2, 1:-1]
        + U2[1:-1, 2:] + U2[1:-1, :-2]
    )

    V1 = U1.copy()
    V2 = U2.copy()
    V1[1:-1, 1:-1] = (1.0 - omega) * U1[1:-1, 1:-1] + omega * avg1
    V2[1:-1, 1:-1] = (1.0 - omega) * U2[1:-1, 1:-1] + omega * avg2
    U1, U2 = V1, V2

    if k in save_steps or k == N_ITER:
        snapshots.append(to_cpu(U1).astype(np.float32))
        steps.append(k)
        interior_maxima.append(float(to_cpu(xp.max(U1[1:-1, 1:-1]))))
        interior_minima.append(float(to_cpu(xp.min(U1[1:-1, 1:-1]))))
        stability_norms.append(float(to_cpu(xp.max(xp.abs(U1 - U2)))))
        residuals.append(residual_inf(U1))

snapshots = np.asarray(snapshots)
steps = np.asarray(steps)
interior_maxima = np.asarray(interior_maxima)
interior_minima = np.asarray(interior_minima)
stability_norms = np.asarray(stability_norms)
residuals = np.asarray(residuals)

print("Mínimo de frontera:", boundary_min_1)
print("Máximo de frontera:", boundary_max_1)
print("Mínimo interior final:", interior_minima[-1])
print("Máximo interior final:", interior_maxima[-1])
print("Norma de la perturbación en la frontera:", boundary_diff_norm)
print("Norma de la diferencia final:", stability_norms[-1])
print("Residuo final:", residuals[-1])

In [ ]:
# ------------------------------------------------------------
# Animación automática
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=FIGSIZE)
im = ax.imshow(
    snapshots[0],
    origin="lower",
    extent=[0.0, 1.0, 0.0, 1.0],
    interpolation="bilinear",
    vmin=boundary_min_1,
    vmax=boundary_max_1,
)
fig.colorbar(im, ax=ax, label=r"$U_{i,j}$")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Relajación armónica y extremos de la solución")
status = ax.text(
    0.02,
    0.98,
    "",
    transform=ax.transAxes,
    va="top",
    bbox={"boxstyle": "round", "alpha": 0.82},
)


def update(frame):
    im.set_data(snapshots[frame])
    status.set_text(
        f"iteración = {steps[frame]}\n"
        f"mínimo interior = {interior_minima[frame]:.6f}\n"
        f"máximo interior = {interior_maxima[frame]:.6f}\n"
        f"mínimo frontera = {boundary_min_1:.6f}\n"
        f"máximo frontera = {boundary_max_1:.6f}"
    )
    return im, status


animation = FuncAnimation(
    fig,
    update,
    frames=len(snapshots),
    interval=1000.0 / FPS,
    blit=False,
)
fig.tight_layout()
save_and_display_animation(
    animation,
    "03.3.A_principio_debil_maximo",
    fps=FPS,
    dpi=DPI,
    bitrate=BITRATE,
)
plt.close(fig)

## Simulación 3.3.B — Estabilidad respecto de los datos de frontera

Sean $u_1$ y $u_2$ soluciones de la misma ecuación de Laplace, pero con datos de frontera $g_1$ y $g_2$.
La estimación que se desea ilustrar es

$$
\max_{\overline\Omega}|u_1-u_2|
\le
\max_{\partial\Omega}|g_1-g_2|.
$$

La gráfica siguiente compara ambas cantidades durante la relajación y muestra el mapa final de
$|u_1-u_2|$.

In [ ]:
# ------------------------------------------------------------
# Gráfica de estabilidad
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.0, 5.5))
ax.semilogx(steps + 1, stability_norms, marker="o", ms=3,
            label=r"$\|U_1-U_2\|_{\infty}$")
ax.axhline(boundary_diff_norm, linestyle="--",
           label=r"$\|g_1-g_2\|_{\infty,\partial\Omega}$")
ax.set_xlabel("iteración + 1")
ax.set_ylabel("norma supremo")
ax.set_title("Estabilidad numérica frente a una perturbación de frontera")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
path = FIG_DIR / "03.3.B_estabilidad_norma_supremo.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)

D_final = np.abs(to_cpu(U1 - U2))
fig, ax = plt.subplots(figsize=(8.0, 6.0))
im = ax.imshow(
    D_final,
    origin="lower",
    extent=[0.0, 1.0, 0.0, 1.0],
    interpolation="bilinear",
    vmin=0.0,
    vmax=boundary_diff_norm,
)
fig.colorbar(im, ax=ax, label=r"$|U_1-U_2|$")
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.set_title("Diferencia entre las dos extensiones armónicas")
fig.tight_layout()
path2 = FIG_DIR / "03.3.B_mapa_diferencia.png"
fig.savefig(path2, dpi=220)
plt.show()
plt.close(fig)

print(path.resolve())
print(path2.resolve())

## Simulación 3.3.C — El máximo interior de una función armónica no constante

Consideramos la función armónica

$$
u(x,y)=e^x\cos y
$$

sobre el disco unitario. Se compara su máximo en discos concéntricos de radio $r<1$ con el máximo sobre la frontera del disco unitario.
La visualización muestra que los máximos aumentan al acercarse a la frontera y que una función armónica no constante no alcanza el máximo global en el interior.

In [ ]:
# ------------------------------------------------------------
# Máximos en discos concéntricos
# ------------------------------------------------------------
radii = np.linspace(0.05, 1.0, 80)
theta = np.linspace(0.0, 2.0 * math.pi, 12000, endpoint=False)
max_on_circles = []
min_on_circles = []

for r in radii:
    values = np.exp(r * np.cos(theta)) * np.cos(r * np.sin(theta))
    max_on_circles.append(np.max(values))
    min_on_circles.append(np.min(values))

max_on_circles = np.asarray(max_on_circles)
min_on_circles = np.asarray(min_on_circles)

fig, ax = plt.subplots(figsize=(9.0, 5.5))
ax.plot(radii, max_on_circles, label=r"$\max_{\partial B_r}u$")
ax.plot(radii, min_on_circles, label=r"$\min_{\partial B_r}u$")
ax.set_xlabel(r"$r$")
ax.set_ylabel("extremo sobre la circunferencia")
ax.set_title("Extremos de una función armónica en círculos concéntricos")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
path = FIG_DIR / "03.3.C_maximo_fuerte_circulos.png"
fig.savefig(path, dpi=220)
plt.show()
plt.close(fig)
print(path.resolve())

# 3.3.1 Principio débil del máximo

#### **Transcripción literal de las notas**

**Principio del Máximo. Teorema I.**

Sea $\Omega\subset\mathbb R^n$ abierto y acotado. Sea $u$ subarmónica en $\Omega$ y supóngase que

$$
\sup_{\Omega}u<\infty.
$$

Entonces

$$
\sup_{\Omega}u
\le
\sup_{\partial\Omega}u.
$$

**Caso.** Sea $\Omega\subset\mathbb R^n$ abierto y acotado y sea $u$ armónica. Entonces

$$
\sup_{\Omega}u
=
\sup_{\partial\Omega}u.
$$

### **Corrección editorial**

Para que el valor de $u$ sobre $\partial\Omega$ esté definido punto por punto, se supone

$$
u\in C(\overline\Omega).
$$

Con esta hipótesis, la formulación natural utiliza el máximo sobre $\overline\Omega$.

## **Teorema 3.3.1 (Principio débil del máximo para funciones subarmónicas).**

Sea $\Omega\subset\mathbb R^n$ un dominio abierto y acotado. Supongamos que

$$
u\in C^2(\Omega)\cap C(\overline\Omega)
$$

y que

$$
\Delta u\ge0
\qquad\text{en }\Omega.
$$

Entonces

$$
\max_{\overline\Omega}u
=
\max_{\partial\Omega}u.
$$

Equivalentemente,

$$
u(x)
\le
\max_{\partial\Omega}u
$$

para todo $x\in\Omega$.

### Demostración

Fijemos $\varepsilon>0$ y definamos

$$
u_\varepsilon(x)=u(x)+\varepsilon|x|^2.
$$

Entonces

$$
\Delta u_\varepsilon
=
\Delta u+2n\varepsilon
>
0
$$

en $\Omega$.

Afirmamos que $u_\varepsilon$ no puede alcanzar un máximo en un punto interior $x_0\in\Omega$.
En efecto, si $x_0$ fuera un punto de máximo interior, la prueba de la segunda derivada daría

$$
D^2u_\varepsilon(x_0)\le0
$$

como forma cuadrática. En particular,

$$
\Delta u_\varepsilon(x_0)
=
\operatorname{tr}D^2u_\varepsilon(x_0)
\le0,
$$

lo cual contradice $\Delta u_\varepsilon>0$.

Por compacidad de $\overline\Omega$ y continuidad de $u_\varepsilon$, su máximo se alcanza; como no puede alcanzarse en el interior, se alcanza en la frontera. Por tanto, para cada $x\in\Omega$,

$$
u(x)+\varepsilon|x|^2
\le
\max_{y\in\partial\Omega}
\bigl(u(y)+\varepsilon|y|^2\bigr).
$$

Como $\Omega$ es acotado, existe $R>0$ tal que $|y|\le R$ para todo $y\in\overline\Omega$. Así,

$$
u(x)
\le
\max_{\partial\Omega}u+\varepsilon R^2.
$$

Haciendo $\varepsilon\downarrow0$ obtenemos

$$
u(x)
\le
\max_{\partial\Omega}u.
$$

Como la frontera está contenida en la clausura,

$$
\max_{\partial\Omega}u
\le
\max_{\overline\Omega}u.
$$

Ambas desigualdades prueban la igualdad.

$\square$

## **Corolario 3.3.2 (Principio débil del mínimo para funciones superarmónicas).**

Sea $\Omega\subset\mathbb R^n$ abierto y acotado y sea

$$
u\in C^2(\Omega)\cap C(\overline\Omega).
$$

Si

$$
\Delta u\le0
\qquad\text{en }\Omega,
$$

entonces

$$
\min_{\overline\Omega}u
=
\min_{\partial\Omega}u.
$$

### Demostración

La función $-u$ satisface

$$
\Delta(-u)\ge0.
$$

Aplicando el Teorema 3.3.1 a $-u$ y multiplicando por $-1$ se obtiene la afirmación.

$\square$

## **Corolario 3.3.3 (Principio débil del máximo para funciones armónicas).**

Bajo las mismas hipótesis, si

$$
\Delta u=0
\qquad\text{en }\Omega,
$$

entonces

$$
\max_{\overline\Omega}|u|
=
\max_{\partial\Omega}|u|.
$$

### Aclaración

La igualdad para $|u|$ se deduce aplicando el principio débil a $u$ y a $-u$; no se está afirmando que $|u|$ sea armónica.

### Ejercicios — Sección 3.3.1

1. Sea $u\in C^2(\Omega)\cap C(\overline\Omega)$ y suponga que $\Delta u>0$ en $\Omega$. Demuestre directamente que $u$ no puede alcanzar un máximo interior.

2. Demuestre el principio débil del máximo usando la propiedad del promedio, sin introducir $u_\varepsilon$.

3. Sea $\Omega$ acotado y sea $u\in C^2(\Omega)\cap C(\overline\Omega)$ tal que
   $$
   -\Delta u\le0
   $$
   en $\Omega$ y $u\le0$ sobre $\partial\Omega$. Determine el signo de $u$ en $\Omega$.

4. **Tipo Examen General.** Sea $\Omega\subset\mathbb R^n$ abierto, acotado y conexo. Suponga que
   $$
   \Delta u\ge -M
   $$
   en $\Omega$, donde $M\ge0$, y que $\Omega\subset B_R(0)$. Construya una función auxiliar cuadrática y obtenga una cota superior para $u$ en términos de $M$, $R$ y $\max_{\partial\Omega}u$.

5. **Tipo Examen General.** Sea $u$ armónica en el exterior de una bola. Use barreras radiales y el principio del máximo para establecer unicidad bajo una condición apropiada de decaimiento en el infinito, distinguiendo los casos $n=2$ y $n\ge3$.

# 3.3.2 Unicidad y estabilidad

#### **Transcripción literal de las notas**

**Unicidad.**

**Estabilidad.** Sean $\Omega\subset\mathbb R^n$ abierto y acotado y $f\in C(\Omega)$. Sean $u_1,u_2$ soluciones de

$$
-\Delta u=f
$$

en $\Omega$, con datos $g_1,g_2\in C(\partial\Omega)$. Entonces

$$
\left.u_i\right|_{\partial\Omega}=g_i,
\qquad i=1,2,
$$

e

$$
|u_1(x)-u_2(x)|
\le
\max_{\partial\Omega}|g_1-g_2|,
\qquad x\in\Omega.
$$

## **Teorema 3.3.4 (Unicidad del problema de Dirichlet para Poisson).**

Sea $\Omega\subset\mathbb R^n$ un dominio abierto y acotado. Sean

$$
f\in C(\Omega),
\qquad
g\in C(\partial\Omega).
$$

Supongamos que

$$
u_1,u_2\in C^2(\Omega)\cap C(\overline\Omega)
$$

satisfacen

$$
\begin{cases}
-\Delta u_i=f, & \text{en }\Omega,\\
u_i=g, & \text{sobre }\partial\Omega,
\end{cases}
\qquad i=1,2.
$$

Entonces

$$
u_1=u_2
$$

en $\overline\Omega$.

### Demostración

Definimos

$$
w=u_1-u_2.
$$

Entonces

$$
\Delta w=0
$$

en $\Omega$ y

$$
w=0
$$

sobre $\partial\Omega$. Por el Corolario 3.3.3,

$$
\max_{\overline\Omega}|w|
=
\max_{\partial\Omega}|w|
=0.
$$

Por tanto $w\equiv0$, es decir, $u_1=u_2$.

$\square$

## **Teorema 3.3.5 (Estabilidad respecto de los datos de Dirichlet).**

Sea $\Omega\subset\mathbb R^n$ un dominio abierto y acotado. Sean

$$
f\in C(\Omega),
\qquad
g_1,g_2\in C(\partial\Omega),
$$

y supongamos que

$$
u_1,u_2\in C^2(\Omega)\cap C(\overline\Omega)
$$

satisfacen

$$
\begin{cases}
-\Delta u_i=f, & \text{en }\Omega,\\
u_i=g_i, & \text{sobre }\partial\Omega,
\end{cases}
\qquad i=1,2.
$$

Entonces

$$
\max_{\overline\Omega}|u_1-u_2|
\le
\max_{\partial\Omega}|g_1-g_2|.
$$

De hecho, bajo estas hipótesis se tiene igualdad:

$$
\max_{\overline\Omega}|u_1-u_2|
=
\max_{\partial\Omega}|g_1-g_2|.
$$

### Demostración

La diferencia

$$
w=u_1-u_2
$$

es armónica y satisface

$$
w=g_1-g_2
$$

sobre la frontera. El Corolario 3.3.3 implica

$$
\max_{\overline\Omega}|w|
=
\max_{\partial\Omega}|w|.
$$

Sustituyendo $w=u_1-u_2$ se obtiene el resultado.

$\square$

### Observación 3.3.1

La estimación expresa dependencia continua en norma supremo respecto de los datos de frontera. La constante de estabilidad es exactamente uno y no depende de la geometría del dominio.

### Ejercicios — Sección 3.3.2

1. Demuestre la unicidad para el problema de Dirichlet de Laplace usando por separado el máximo de $w$ y el máximo de $-w$.

2. Sean $u_1,u_2$ soluciones de $-\Delta u_i=f_i$ con datos $g_i$. Construya una función auxiliar que permita controlar $u_1-u_2$ cuando $f_1\ne f_2$ y el dominio esté contenido en una bola.

3. Demuestre que la aplicación que asigna a cada dato de frontera continuo su extensión armónica es lineal y contractiva en la norma supremo.

4. **Tipo Examen General.** Sea $B_1(0)\subset\mathbb R^n$. Suponga que
   $$
   -\Delta u=f
   $$
   en $B_1(0)$ y $u=g$ sobre $\partial B_1(0)$. Use una barrera cuadrática y el principio del máximo para demostrar una estimación de la forma
   $$
   \max_{\overline{B_1(0)}}|u|
   \le
   C_n\left(
   \max_{\overline{B_1(0)}}|f|
   +
   \max_{\partial B_1(0)}|g|
   \right).
   $$

5. **Tipo Examen General.** Sea $L=\Delta+c$ en un dominio acotado. Investigue para qué signos de $c$ el principio del máximo proporciona unicidad con datos de Dirichlet nulos y construya un contraejemplo cuando la condición de signo no sea válida.

# 3.3.3 Principio fuerte del máximo

#### **Transcripción literal de las notas**

**Teorema II.** Sea $\Omega\subset\mathbb R^n$ abierto, conexo y acotado. Sea $u$ una función que tiene la propiedad del promedio. Si existe $x_0\in\Omega$ tal que

$$
u(x_0)=\max_{\overline\Omega}u,
$$

entonces $u$ es constante.

En el comienzo de la página siguiente aparece la consecuencia de positividad para el problema de Dirichlet: si el dato de frontera es no negativo y es positivo en algún punto de la frontera, entonces la solución armónica es positiva en todo el interior.

## **Teorema 3.3.6 (Principio fuerte del máximo para funciones armónicas).**

Sea $\Omega\subset\mathbb R^n$ un dominio abierto y conexo y sea $u\in C^2(\Omega)$ armónica. Si existe $x_0\in\Omega$ tal que

$$
u(x_0)=\sup_{x\in\Omega}u(x),
$$

entonces $u$ es constante en $\Omega$.

### Demostración mediante la propiedad del promedio

Sea

$$
M=u(x_0).
$$

Definimos

$$
A=\{x\in\Omega:u(x)=M\}.
$$

El conjunto $A$ es no vacío porque $x_0\in A$. Además, $A$ es cerrado en $\Omega$ por continuidad de $u$.

Probaremos que $A$ es abierto. Sea $x\in A$. Como $\Omega$ es abierto, existe $r>0$ tal que

$$
\overline{B_r(x)}\subset\Omega.
$$

La propiedad del promedio da

$$
M=u(x)
=
\frac{1}{|B_r(x)|}
\int_{B_r(x)}u(y)\,dy.
$$

Como $u(y)\le M$ para todo $y\in\Omega$, la función continua no negativa

$$
M-u(y)
$$

tiene integral cero sobre $B_r(x)$. Por consiguiente,

$$
u(y)=M
$$

para todo $y\in B_r(x)$. Así, $B_r(x)\subset A$ y $A$ es abierto.

El conjunto $A$ es abierto, cerrado y no vacío dentro del conjunto conexo $\Omega$. Por tanto,

$$
A=\Omega.
$$

Luego $u\equiv M$ en $\Omega$.

$\square$

## **Corolario 3.3.7 (Principio fuerte del mínimo).**

Sea $u$ armónica en un dominio conexo $\Omega$. Si $u$ alcanza su ínfimo en un punto interior, entonces $u$ es constante.

## **Corolario 3.3.8 (Positividad estricta de la extensión armónica).**

Sea $\Omega\subset\mathbb R^n$ un dominio abierto, acotado y conexo. Supongamos que

$$
u\in C^2(\Omega)\cap C(\overline\Omega)
$$

satisface

$$
\begin{cases}
\Delta u=0, & \text{en }\Omega,\\
u=g, & \text{sobre }\partial\Omega,
\end{cases}
$$

donde $g\in C(\partial\Omega)$ cumple

$$
g\ge0
$$

sobre $\partial\Omega$ y $g$ no es idénticamente cero. Entonces

$$
u(x)>0
$$

para todo $x\in\Omega$.

### Demostración

El principio débil del mínimo implica

$$
u\ge0
$$

en $\overline\Omega$. Si existiera $x_0\in\Omega$ tal que $u(x_0)=0$, entonces $u$ alcanzaría su mínimo en un punto interior. Por el principio fuerte del mínimo, $u$ sería constante e igual a cero. Esto contradice que $g$ no sea idénticamente cero.

Por tanto, $u>0$ en $\Omega$.

$\square$

### Aclaración

En las notas se expresa la no trivialidad del dato mediante la existencia de un punto de frontera donde $g>0$. Como $g$ es continua y $g\ge0$, esto equivale a afirmar que $g$ no es idénticamente cero.

### Ejercicios — Sección 3.3.3

1. Demuestre el principio fuerte del máximo usando promedios sobre esferas en lugar de promedios sobre bolas.

2. Sea $u$ armónica y no constante en un dominio conexo. Demuestre que
   $$
   \min_{\partial\Omega}u
   <
   u(x)
   <
   \max_{\partial\Omega}u
   $$
   para todo $x\in\Omega$, suponiendo $u\in C(\overline\Omega)$ y $\Omega$ acotado.

3. Sea $g\ge0$ sobre $\partial\Omega$. Explique por qué la conclusión $u>0$ en el interior puede fallar si $\Omega$ no es conexo y el dato es cero en la frontera de una componente.

4. **Tipo Examen General.** Sea $u$ armónica y acotada en todo $\mathbb R^n$. Utilice el principio fuerte del máximo junto con una estimación o una propiedad adicional apropiada para demostrar que $u$ es constante. Indique con claridad dónde se utiliza la acotación global.

5. **Tipo Examen General.** Sea $R=(0,\pi)^2$ y suponga que $u\in C^2(R)\cap C^1(\overline R)$ es armónica y satisface
   $$
   u(x,0)=u_y(x,0)=0
   $$
   para $0<x<\pi$. Diseñe una estrategia rigurosa para demostrar $u\equiv0$, explicando por qué el principio del máximo por sí solo no usa directamente el dato de Neumann.

# Control de cobertura y estado del capítulo

## Cubierto en este notebook

- principio débil del máximo para funciones subarmónicas;
- principio débil del mínimo para funciones superarmónicas;
- principio del máximo para funciones armónicas;
- unicidad del problema de Dirichlet para Poisson;
- estabilidad en norma supremo respecto de los datos de frontera;
- principio fuerte del máximo y del mínimo;
- positividad estricta para datos de frontera no negativos y no triviales;
- simulaciones del máximo, la estabilidad y la propagación de extremos hacia la frontera.

## Contenido añadido para cerrar huecos

- perturbación estrictamente subarmónica $u_\varepsilon=u+\varepsilon|x|^2$;
- formulación precisa con continuidad hasta la frontera;
- demostraciones completas de unicidad y estabilidad;
- demostración topológica del principio fuerte mediante la propiedad del promedio;
- aclaración de la hipótesis de no trivialidad del dato de frontera.

## Pendiente inmediato

El siguiente bloque de la página 7 contiene:

- teorema de comparación;
- subsoluciones y supersoluciones;
- operadores de la forma $\Delta u+h(x)u=f$;
- condiciones de signo sobre el término de orden cero.

El siguiente notebook será

`03.3.02_Comparacion_sub_y_supersoluciones.ipynb`.